## Cell 0 — GPU selection

Pick which physical GPUs this notebook is allowed to use. We set `CUDA_VISIBLE_DEVICES` *before* any `import torch` so PyTorch sees only the chosen devices — once torch is imported, changing this env var has no effect for the rest of the process. After this cell, the visible devices are renumbered starting at 0 from torch's point of view, so `device_map="auto"` will only shard across the ones we picked.

In [ ]:
import os

# Choose any 6 of the 8 physical GPU indices (0..7). Edit this list to taste.
VISIBLE_GPUS = [0, 1, 2, 3, 4, 5]
assert "torch" not in globals(), (
    "torch is already imported — restart the kernel before changing VISIBLE_GPUS, "
    "because CUDA_VISIBLE_DEVICES is read once at torch import time."
)
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in VISIBLE_GPUS)
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

# GPT-OSS-120B Output Format Inspection

Goal: see exactly how GPT-OSS-120B formats its output — every special token, every channel boundary — so SFT training data can match the format precisely and avoid reasoning collapse.

Constraints:
- `transformers` only (no `openai-harmony`, no `gpt_oss` helper).
- torch 2.6.0, bf16, `device_map="auto"`. No quantization, no DeepSpeed.
- Local weights only. No network calls.

## Cell 1 — Setup and config
Print versions and GPU topology so this notebook is self-documenting when re-run on a different node.

In [ ]:
import json
import pprint
import re

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

# Set this to wherever your local GPT-OSS-120B checkpoint lives.
# All other cells assume MODEL_PATH points at a directory containing config.json,
# tokenizer files, and the sharded safetensors.
MODEL_PATH = "/path/to/gpt-oss-120b"

print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"cuda devices: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    total_gb = props.total_memory / (1024 ** 3)
    print(f"  [{i}] {props.name}  total={total_gb:.1f} GiB")

## Cell 2 — Tokenizer and special tokens

The tokenizer is the source of truth for which special tokens exist and what their IDs are. The `chat_template` (Jinja) is the source of truth for how messages are assembled — more authoritative than any blog post, because it is exactly what was used during the model's post-training. We must match it byte-for-byte in our SFT data.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

print("===== special_tokens_map =====")
pprint.pprint(tokenizer.special_tokens_map)

print("\n===== additional_special_tokens =====")
pprint.pprint(tokenizer.additional_special_tokens)

# Full list of added tokens (special + user-added), sorted by id so the layout
# of the special-token block is obvious.
added_vocab = tokenizer.get_added_vocab()
added_sorted = sorted(added_vocab.items(), key=lambda kv: kv[1])
print("\n===== get_added_vocab() (sorted by id) =====")
for tok, tid in added_sorted:
    print(f"  {tid:>7d}  {tok!r}")

In [ ]:
# Look up the Harmony-format tokens explicitly. If a name doesn't exist in this
# tokenizer (different fork / renamed), we still want to know — so we warn loudly
# rather than silently skipping. Anything we miss here is something we'd risk
# tokenizing wrong in SFT data.
HARMONY_TOKENS = [
    "<|start|>", "<|end|>", "<|message|>", "<|channel|>",
    "<|return|>", "<|call|>", "<|constrain|>",
]
ROLE_TOKENS = ["system", "developer", "user", "assistant", "tool"]

print("===== Harmony framing tokens =====")
for t in HARMONY_TOKENS:
    tid = tokenizer.convert_tokens_to_ids(t)
    if tid is None or tid == tokenizer.unk_token_id:
        # Fall back to substring search across the added vocab.
        candidates = [(k, v) for k, v in added_vocab.items() if t.strip("<|>") in k]
        print(f"  {t!r:>16s} -> NOT FOUND.  closest: {candidates[:5]}")
    else:
        print(f"  {t!r:>16s} -> id={tid}")

print("\n===== Role tokens (as bare strings — these may be plain text, not specials) =====")
for r in ROLE_TOKENS:
    ids = tokenizer.encode(r, add_special_tokens=False)
    print(f"  {r!r:>12s} -> ids={ids}  decoded={tokenizer.decode(ids)!r}")

In [ ]:
# The chat template is the ground truth for SFT data layout.
print("===== tokenizer.chat_template (Jinja) =====")
ct = tokenizer.chat_template
if ct is None:
    print("WARNING: no chat_template attached to this tokenizer.")
else:
    print(ct)

## Cell 3 — Load the model

Weights on disk are MXFP4. `AutoModelForCausalLM` will materialize compute in bf16 — that's expected for inspection; we are not optimizing inference cost here. `device_map="auto"` lets HF Accelerate shard across whatever GPUs we have.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

cfg = model.config
interesting = [
    "model_type", "num_hidden_layers", "num_attention_heads",
    "num_key_value_heads", "hidden_size", "vocab_size",
    "num_local_experts", "num_experts_per_tok",
    "sliding_window", "rope_scaling", "max_position_embeddings",
]
print("===== model.config (selected) =====")
for k in interesting:
    if hasattr(cfg, k):
        print(f"  {k}: {getattr(cfg, k)}")

print("\n===== hf_device_map =====")
if hasattr(model, "hf_device_map"):
    pprint.pprint(model.hf_device_map)
else:
    print("(no hf_device_map; model fit on a single device)")

## Cell 4 — Inspection helpers

Three small, flat helpers — no class hierarchy, deliberately. The point of this notebook is that every step is inspectable.

In [ ]:
def apply_template_and_show(messages, reasoning_effort=None):
    """Render messages through the chat template and print the raw string.

    We need add_generation_prompt=True so the rendered text ends right where the
    assistant turn would begin — that is the actual prompt the model sees at
    inference time.
    """
    kwargs = dict(add_generation_prompt=True, tokenize=False)
    if reasoning_effort is not None:
        # Some forks accept this as a template kwarg; if the template ignores it,
        # nothing breaks — we'd just see the same prompt regardless of effort.
        kwargs["reasoning_effort"] = reasoning_effort
    try:
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError:
        # Template doesn't accept reasoning_effort — drop and retry.
        kwargs.pop("reasoning_effort", None)
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
    print("===== RENDERED PROMPT (specials visible) =====")
    print(rendered)
    print("===== END RENDERED PROMPT =====")
    return rendered


def tokenize_and_show(text, head=200, tail=50):
    """Tokenize and dump (id, repr) per token. Looking at repr makes whitespace
    and special-token byte-shapes visible — decode() alone hides them."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = tokenizer.convert_ids_to_tokens(ids)
    pairs = list(zip(ids, pieces))
    print(f"===== TOKENS (total={len(pairs)}) — first {head} =====")
    for tid, p in pairs[:head]:
        print(f"  {tid:>7d}  {p!r}")
    if len(pairs) > head + tail:
        print(f"  ... ({len(pairs) - head - tail} omitted) ...")
        print(f"===== TOKENS — last {tail} =====")
        for tid, p in pairs[-tail:]:
            print(f"  {tid:>7d}  {p!r}")
    return pairs

In [ ]:
# Channel parser: scans the raw assistant output and splits it into channel
# segments. Harmony format puts content in blocks like
#   <|channel|>analysis<|message|>...<|end|>
#   <|channel|>final<|message|>...<|return|>
# We don't assume the closing token — <|end|>, <|return|>, or <|call|> can all
# terminate a channel — so we close on whichever comes first.
_CHANNEL_RE = re.compile(
    r"<\|channel\|>(?P<channel>[^<]+?)<\|message\|>(?P<content>.*?)(?=<\|end\|>|<\|return\|>|<\|call\|>|<\|channel\|>|\Z)",
    re.DOTALL,
)


def parse_channels(raw_text):
    segments = []
    for m in _CHANNEL_RE.finditer(raw_text):
        segments.append({
            "channel": m.group("channel").strip(),
            "content": m.group("content"),
        })
    if not segments:
        # If the regex finds nothing, surface that — better a loud warning than a
        # silent empty list that hides a format mismatch.
        print("WARNING: no <|channel|>...<|message|>...<terminator> blocks found.")
        print("         Inspect the raw output above and adjust the regex.")
    return segments


def generate_and_dissect(messages, max_new_tokens=2048, reasoning_effort="medium",
                         temperature=1.0, top_p=1.0):
    """Render -> tokenize -> generate -> decode WITH specials -> parse channels."""
    rendered = apply_template_and_show(messages, reasoning_effort=reasoning_effort)
    inputs = tokenizer(rendered, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0 and temperature != 1.0) or top_p < 1.0,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    with torch.no_grad():
        out = model.generate(**inputs, generation_config=gen_cfg)

    # Slice off the prompt — only the generated suffix is interesting for format.
    gen_ids = out[0, input_len:].tolist()

    # CRITICAL: skip_special_tokens=False. Without this we lose the entire point
    # of the exercise.
    raw_decoded = tokenizer.decode(gen_ids, skip_special_tokens=False)
    clean_decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)

    print("\n===== RAW OUTPUT (specials visible) =====")
    print(raw_decoded)
    print("===== END RAW OUTPUT =====")

    print("\n===== CLEAN OUTPUT (specials stripped — for comparison only) =====")
    print(clean_decoded)
    print("===== END CLEAN OUTPUT =====")

    head_pieces = tokenizer.convert_ids_to_tokens(gen_ids[:100])
    head_pairs = list(zip(gen_ids[:100], head_pieces))
    print("\n===== FIRST 100 GENERATED TOKENS (id, repr) =====")
    for tid, p in head_pairs:
        print(f"  {tid:>7d}  {p!r}")

    segments = parse_channels(raw_decoded)
    print("\n===== PARSED CHANNEL SEGMENTS =====")
    for i, seg in enumerate(segments):
        preview = seg["content"][:200].replace("\n", "\\n")
        print(f"  [{i}] channel={seg['channel']!r}  len={len(seg['content'])}  preview={preview!r}")

    return {
        "raw": raw_decoded,
        "clean": clean_decoded,
        "gen_ids": gen_ids,
        "head_pairs": head_pairs,
        "segments": segments,
    }

## Cell 5 — Controlled experiment: Amex sales-call conversion prediction

Realistic surrogate of the actual SFT use case. We sweep `reasoning_effort` across `low`, `medium`, `high` to see how the channel structure (and the analysis/final token ratio) shifts.

In [ ]:
SYSTEM_PROMPT = """You are an expert sales analyst evaluating outbound calls. 
Given a call transcript, predict whether the call will result in a successful 
conversion (customer accepting the offer). Respond with your reasoning and 
then a final answer of either CONVERT or NO_CONVERT."""

CONTEXT_TRANSCRIPT = """[Agent]: Hi, this is Sarah from American Express. Am I 
speaking with Mr. Johnson?
[Customer]: Yes, this is he. What is this about?
[Agent]: I'm calling about a pre-approved offer for our Platinum card with a 
$200 statement credit after your first purchase...
[Customer]: I already have a Platinum card. I've had it for years.
[Agent]: Oh, I see. Well, we have an upgrade path that could give you 
additional benefits...
[Customer]: I'm not interested in changing anything right now. I'm happy with 
what I have.
[Agent]: I understand. Could I at least send you some information by email?
[Customer]: Sure, fine. Send the email. I have to go now.
[Agent]: Thank you, have a great day."""

USER_PROMPT = f"Analyze the following call transcript and predict conversion outcome.\n\n{CONTEXT_TRANSCRIPT}"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

In [ ]:
results = {}
for effort in ["low", "medium", "high"]:
    print("\n")
    print("#" * 80)
    print(f"# REASONING EFFORT = {effort}")
    print("#" * 80)
    res = generate_and_dissect(
        messages,
        max_new_tokens=2048,
        reasoning_effort=effort,
        temperature=1.0,
        top_p=1.0,
    )

    # Per-channel token counts. We re-tokenize the parsed content (not the raw
    # block, so the framing tokens don't pollute the count) — this is what we'd
    # actually compute loss over in SFT.
    per_channel_tokens = {}
    for seg in res["segments"]:
        n = len(tokenizer.encode(seg["content"], add_special_tokens=False))
        per_channel_tokens[seg["channel"]] = per_channel_tokens.get(seg["channel"], 0) + n
    res["per_channel_tokens"] = per_channel_tokens
    res["total_gen_tokens"] = len(res["gen_ids"])

    print("\n===== TOKEN COUNTS PER CHANNEL =====")
    for ch, n in per_channel_tokens.items():
        print(f"  {ch:>20s}: {n}")
    print(f"  {'TOTAL (incl. specials)':>20s}: {res['total_gen_tokens']}")

    final_only = "".join(s["content"] for s in res["segments"] if s["channel"] == "final")
    res["final_only"] = final_only.strip()
    print("\n===== FINAL CHANNEL ONLY (clean) =====")
    print(res["final_only"])
    print("===== END FINAL =====")

    results[effort] = res

## Cell 6 — Comparison summary

Quick at-a-glance read of how reasoning effort scales the analysis channel relative to the final channel. The ratio matters: it tells you what fraction of an SFT example's tokens come from reasoning vs. answer, which feeds directly into the loss-masking decision.

In [ ]:
def extract_verdict(text):
    # Cheap pattern match — the prompt asks for CONVERT or NO_CONVERT. We look
    # for the last occurrence so any echo in the reasoning doesn't fool us.
    matches = re.findall(r"NO_CONVERT|CONVERT", text or "")
    return matches[-1] if matches else "(none)"

header = f"{'effort':<8} {'total':>8} {'analysis':>10} {'final':>8} {'a/f ratio':>10}  verdict"
print(header)
print("-" * len(header))
for effort, res in results.items():
    total = res["total_gen_tokens"]
    a = res["per_channel_tokens"].get("analysis", 0)
    f = res["per_channel_tokens"].get("final", 0)
    ratio = (a / f) if f else float("inf")
    verdict = extract_verdict(res["final_only"])
    print(f"{effort:<8} {total:>8d} {a:>10d} {f:>8d} {ratio:>10.2f}  {verdict}")

## Cell 7 — Notes for SFT data construction

*(Fill in after running cells 2 and 5. The observations above tell you what the format actually is; this cell turns those observations into training-data decisions.)*

### Special tokens to preserve verbatim in SFT targets
- _List the exact token IDs from cell 2 that must appear unmodified in every training example (e.g., `<|start|>`, `<|channel|>`, `<|message|>`, `<|end|>`, `<|return|>`)._

### Channel structure observed
- _What channels appeared in cell 5 (e.g., `analysis`, `commentary`, `final`)? In what order? Did any effort level skip a channel?_
- _Did the prompt rendered in cell 4 already contain channel framing, or only the assistant continuation?_

### Loss masking strategy implications
- _Should we compute loss on:_
  - _**(a)** final channel only — safest against reasoning collapse, but loses the steering signal._
  - _**(b)** both channels at full weight — highest collapse risk because analysis is much longer than final._
  - _**(c)** both channels with analysis weighted ~0.1–0.3x — reasonable middle path for a pattern-recognition task in a regulated environment._
- _Token-count ratio from cell 6 directly informs the down-weighting factor: if analysis is 10x final, weighting analysis at 0.1 makes per-channel contributions roughly comparable._

### Format risks if Harmony structure is broken
- _If we forget the `<|channel|>final<|message|>` framing, the model may keep generating analysis indefinitely._
- _If we mix special tokens with their literal-string equivalents (e.g., the string `"<|end|>"` instead of the special-token id), the tokenizer will encode them as multiple BPE pieces and the model will not recognize the boundary._
- _If we strip `<|return|>` vs `<|end|>` distinctions, we may lose stop-token semantics during generation._